In [1]:
import numpy as np
import pandas as pd
import logging
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from flaml import AutoML

# Configure Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def load_data(file_path):
    """Loads CSV and performs initial drops."""
    df = pd.read_csv(file_path)
    if 'url' in df.columns:
        df.drop(columns=['url'], inplace=True)
    logging.info(f"Data loaded. Shape: {df.shape}")
    return df

def clean_features(X_train, X_test):
    """Removes constant columns and zero-variance (IQR=0) columns."""
    # Constant columns
    constant_cols = [col for col in X_train.columns if X_train[col].nunique() <= 1]
    
    # IQR = 0 columns
    iqr_zero_cols = []
    for col in X_train.select_dtypes(include=['int64', 'float64']).columns:
        if X_train[col].quantile(0.75) - X_train[col].quantile(0.25) == 0:
            iqr_zero_cols.append(col)
            
    drop_cols = list(set(constant_cols + iqr_zero_cols))
    logging.info(f"Dropping {len(drop_cols)} low-variance columns.")
    
    return X_train.drop(columns=drop_cols), X_test.drop(columns=drop_cols)

def get_preprocessing_pipeline(X_train):
    """Creates the column transformer for numeric and categorical data."""
    num_cols = X_train.select_dtypes(exclude='object').columns
    cat_cols = X_train.select_dtypes(include='object').columns

    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', MinMaxScaler())
    ])

    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    return ColumnTransformer([
        ('num', num_pipe, num_cols),
        ('cat', cat_pipe, cat_cols)
    ])

def apply_resampling_and_pca(X_train_proc, y_train):
    """Applies SMOTE and PCA to the preprocessed training data."""
    smote = SMOTE(random_state=42)
    X_res, y_res = smote.fit_resample(X_train_proc, y_train)
    
    pca = PCA(n_components=0.95, random_state=42)
    X_pca = pca.fit_transform(X_res)
    
    logging.info(f"PCA reduced features to: {X_pca.shape[1]}")
    return X_pca, y_res, pca

def run_automl(X_train, y_train, budget=300):
    """Initializes and fits FLAML AutoML."""
    automl = AutoML()
    settings = {
        "time_budget": budget,
        "metric": "accuracy",
        "task": "classification",
        "estimator_list": ["rf", "extra_tree", "xgboost", "lrl2"],
        "log_file_name": "flaml.log",
        "seed": 42
    }
    automl.fit(X_train=X_train, y_train=y_train, **settings)
    return automl

def evaluate_model(model, X_test, y_test):
    """Prints evaluation metrics."""
    y_pred = model.predict(X_test)
    print("\n--- Model Performance ---")
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

# --- MAIN EXECUTION FLOW ---

if __name__ == "__main__":
    # 1. Load and Split
    data_path = r'C:\Phissing_Prediction_Model\data\raw\Phissing_Dataset.csv'
    df = load_data(data_path)
    
    le = LabelEncoder()
    df['status'] = le.fit_transform(df['status'])
    
    X = df.drop(columns=['status'])
    y = df['status']
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    # 2. Clean and Preprocess
    X_train, X_test = clean_features(X_train, X_test)
    preprocessor = get_preprocessing_pipeline(X_train)
    
    X_train_proc = preprocessor.fit_transform(X_train)
    X_test_proc = preprocessor.transform(X_test)

    # 3. SMOTE and PCA
    X_train_final, y_train_final, fitted_pca = apply_resampling_and_pca(X_train_proc, y_train)
    X_test_final = fitted_pca.transform(X_test_proc)

    # 4. Train and Evaluate
    model = run_automl(X_train_final, y_train_final)
    evaluate_model(model, X_test_final, y_test)

2026-03-08 04:23:32,397 - INFO - Data loaded. Shape: (11430, 88)
2026-03-08 04:23:32,517 - INFO - Dropping 49 low-variance columns.
2026-03-08 04:23:34,366 - INFO - PCA reduced features to: 15


[flaml.automl.logger: 03-08 04:23:34] {2375} INFO - task = classification
[flaml.automl.logger: 03-08 04:23:34] {2386} INFO - Evaluation method: cv
[flaml.automl.logger: 03-08 04:23:34] {2489} INFO - Minimizing error metric: 1-accuracy
[flaml.automl.logger: 03-08 04:23:34] {2606} INFO - List of ML learners in AutoML Run: ['rf', 'extra_tree', 'xgboost', 'lrl2']
[flaml.automl.logger: 03-08 04:23:34] {2911} INFO - iteration 0, current learner rf
[flaml.automl.logger: 03-08 04:23:34] {3046} INFO - Estimated sufficient time budget=2415s. Estimated necessary time budget=4s.
[flaml.automl.logger: 03-08 04:23:34] {3097} INFO -  at 0.2s,	estimator rf's best error=1.6621e-01,	best estimator rf's best error=1.6621e-01
[flaml.automl.logger: 03-08 04:23:34] {2911} INFO - iteration 1, current learner xgboost
[flaml.automl.logger: 03-08 04:23:34] {3097} INFO -  at 0.6s,	estimator xgboost's best error=1.3834e-01,	best estimator xgboost's best error=1.3834e-01
[flaml.automl.logger: 03-08 04:23:34] {291

2026-03-08 04:23:47,499 - INFO - No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 03-08 04:23:47] {3097} INFO -  at 13.2s,	estimator lrl2's best error=9.5600e-02,	best estimator rf's best error=8.0979e-02
[flaml.automl.logger: 03-08 04:23:47] {2911} INFO - iteration 33, current learner lrl2
[flaml.automl.logger: 03-08 04:23:47] {3097} INFO -  at 13.3s,	estimator lrl2's best error=9.5475e-02,	best estimator rf's best error=8.0979e-02
[flaml.automl.logger: 03-08 04:23:47] {2911} INFO - iteration 34, current learner rf


c:\Phissing_Prediction_Model\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Phissing_Prediction_Model\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Phissing_Prediction_Model\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_rati

...

[flaml.automl.logger: 03-08 04:28:34] {3097} INFO -  at 299.7s,	estimator extra_tree's best error=6.8483e-02,	best estimator xgboost's best error=5.6611e-02
[flaml.automl.logger: 03-08 04:28:34] {2911} INFO - iteration 118, current learner lrl2
[flaml.automl.logger: 03-08 04:28:34] {3097} INFO -  at 299.8s,	estimator lrl2's best error=9.5225e-02,	best estimator xgboost's best error=5.6611e-02
[flaml.automl.logger: 03-08 04:28:34] {2911} INFO - iteration 119, current learner lrl2
[flaml.automl.logger: 03-08 04:28:34] {3097} INFO -  at 299.8s,	estimator lrl2's best error=9.5225e-02,	best estimator xgboost's best error=5.6611e-02
[flaml.automl.logger: 03-08 04:28:34] {2911} INFO - iteration 120, current learner lrl2
[flaml.automl.logger: 03-08 04:28:34] {3097} INFO -  at 299.9s,	estimator lrl2's best error=9.5225e-02,	best estimator xgboost's best error=5.6611e-02
[flaml.automl.logger: 03-08 04:28:34] {2911} INFO - iteration 121, current learner lrl2


c:\Phissing_Prediction_Model\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Phissing_Prediction_Model\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Phissing_Prediction_Model\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_rati

[flaml.automl.logger: 03-08 04:28:34] {3097} INFO -  at 300.0s,	estimator lrl2's best error=9.5225e-02,	best estimator xgboost's best error=5.6611e-02


c:\Phissing_Prediction_Model\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Phissing_Prediction_Model\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Phissing_Prediction_Model\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_rati

[flaml.automl.logger: 03-08 04:28:36] {3359} INFO - retrain xgboost for 1.9s
[flaml.automl.logger: 03-08 04:28:36] {3362} INFO - retrained model: XGBClassifier(base_score=None, booster=None, callbacks=[],
              colsample_bylevel=np.float64(0.9925605286749656),
              colsample_bynode=None, colsample_bytree=1.0, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, feature_types=None, feature_weights=None,
              gamma=None, grow_policy='lossguide', importance_type=None,
              interaction_constraints=None,
              learning_rate=np.float64(0.38257652588179264), max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=0, max_leaves=133,
              min_child_weight=np.float64(0.05817750457111067), missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=92,
              n_jobs=-1, num_parallel_tree=